# 4 · Ejercicio 3: listas de cinco formas (35 min)

El Ejercicio 3 casi siempre pide **la misma función** implementada de varias formas. Aquí verás las cinco con **un solo ejemplo**, para que se vea que son traducciones unas de otras.

**Ejemplo:** `evens(l)` devuelve los números pares de `l`, en el mismo orden.

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen.

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    tree match
        case Empty() => empty
        case Node(left, a, right) =>
            node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

In [ ]:
class TestEvens(evens: List[Int] => List[Int]) extends AnyFlatSpec with should.Matchers:
    "evens" should "work" in:
        evens(List()) shouldBe List()
        evens(List(1, 2, 3, 4, 6)) shouldBe List(2, 4, 6)
        evens(List(1, 3)) shouldBe List()

## Forma 1 · Recursión (divide y vencerás)

1. ¿Qué devuelvo para `Nil`?
2. Para `head :: tail`: **supón que ya tienes la solución del resto** (`tailSol`). ¿Cómo la combino con `head`?

In [ ]:
def evens(l: List[Int]): List[Int] =
    l match
        case Nil => Nil
        case head :: tail =>
            val tailSol = evens(tail)
            if head % 2 == 0 then head :: tailSol else tailSol

run(TestEvens(evens))

## Forma 2 · `foldRight` (traducción directa de la forma 1)

`l.foldRight(LO_DE_NIL)((head, tailSol) => LO_DEL_CASO_RECURSIVO)`

| Recursión | `foldRight` |
|---|---|
| `case Nil => X` | primer argumento: `X` |
| cuerpo de `case head :: tail` | la función `(head, tailSol) => ...` |
| `evens(tail)` | `tailSol` |

In [ ]:
def evens(l: List[Int]): List[Int] =
    l.foldRight(Nil: List[Int])((head, tailSol) =>
        if head % 2 == 0 then head :: tailSol else tailSol)

run(TestEvens(evens))

## Forma 3 · Recursión final (patrón iterativo)

Es un bucle: un acumulador `out` que empieza con un valor y se actualiza con cada elemento. La llamada recursiva es **lo último** que se hace.

⚠️ Al añadir con `::` recorriendo de izquierda a derecha, la lista queda **al revés** → `.reverse` al final.

In [ ]:
def evens(l: List[Int]): List[Int] =

    def step(out: List[Int], aux: List[Int]): List[Int] =
        aux match
            case Nil => out
            case head :: tail =>
                step(if head % 2 == 0 then head :: out else out, tail)

    step(Nil, l).reverse

run(TestEvens(evens))

## Forma 4 · `foldLeft` (traducción directa de la forma 3)

`l.foldLeft(VALOR_INICIAL)((out, head) => NUEVO_OUT)` y luego el retoque final.

⚠️ El orden es `(acumulado, elemento)`, **al revés** que en `foldRight`.

In [ ]:
def evens(l: List[Int]): List[Int] =
    l.foldLeft(Nil: List[Int])((out, head) =>
        if head % 2 == 0 then head :: out else out
    ).reverse

run(TestEvens(evens))

## Forma 5 · Sin recursión: funciones de orden superior

| Función | Hace |
|---|---|
| `map(f)` | transforma cada elemento |
| `filter(p)` | se queda con los que cumplen `p` |
| `flatMap(f)` | cada elemento da una lista, y se juntan |
| `exists(p)` / `forall(p)` | ¿alguno / todos cumplen? |
| `zipWithIndex`, `zip`, `sum`, `reverse`… | ver preámbulo del examen |

In [ ]:
def evens(l: List[Int]): List[Int] =
    l.filter(_ % 2 == 0)

def evens2(l: List[Int]): List[Int] =
    l.flatMap(x => if x % 2 == 0 then List(x) else Nil)

run(TestEvens(evens))
run(TestEvens(evens2))

## Dos trucos que salen mucho

**1. Usar `case` dentro de los folds** cuando hay que mirar la forma de los datos. Cada caso recibe el par `(head, tailSol)` (o `(out, head)` en `foldLeft`):

In [ ]:
// split: separa Lefts y Rights
def split[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    l.foldRight((List[A](), List[B]())):
        case (Left(a), (as, bs))  => (a :: as, bs)
        case (Right(b), (as, bs)) => (as, b :: bs)

split(List(Left(1), Right('a'), Left(2)))

**2. Acumulador con información extra (tupla).** Si para decidir necesitas la posición, el elemento anterior, un contador… guárdalo junto a la solución y al final quédate con lo que importa (`._1`).

In [ ]:
// elementos en posiciones pares (0, 2, 4…)
def evenPositions[A](l: List[A]): List[A] =
    l.foldLeft((List[A](), 0)):
        case ((out, i), e) => (if i % 2 == 0 then e :: out else out, i + 1)
    ._1.reverse

evenPositions(List('a', 'b', 'c', 'd', 'e'))

## Mini-ejercicio (10 min): `countIf`

`countIf(p, l)` cuenta cuántos elementos de `l` cumplen `p`. Hazlo de las cinco formas (la quinta con `filter` y `length`).

In [ ]:
class TestCount(countIf: (Int => Boolean, List[Int]) => Int) extends AnyFlatSpec with should.Matchers:
    "countIf" should "work" in:
        countIf(_ > 0, List()) shouldBe 0
        countIf(_ > 0, List(1, -2, 3)) shouldBe 2
        countIf(_ % 2 == 0, List(2, 4, 6)) shouldBe 3

In [ ]:
def countIf[A](p: A => Boolean, l: List[A]): Int =
    ???

run(TestCount(countIf))

<details>
<summary><b>Ver solución</b></summary>

```scala
// 1. recursión
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l match
        case Nil => 0
        case head :: tail => (if p(head) then 1 else 0) + countIf(p, tail)

// 2. foldRight
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l.foldRight(0)((head, tailSol) => if p(head) then tailSol + 1 else tailSol)

// 3. recursión final
def countIf[A](p: A => Boolean, l: List[A]): Int =
    def step(out: Int, aux: List[A]): Int =
        aux match
            case Nil => out
            case head :: tail => step(if p(head) then out + 1 else out, tail)
    step(0, l)

// 4. foldLeft
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l.foldLeft(0)((out, head) => if p(head) then out + 1 else out)

// 5. HOFs
def countIf[A](p: A => Boolean, l: List[A]): Int =
    l.filter(p).length
```

</details>

**Siguiente:** `05-ejercicio4-patrones.ipynb`